# Isoflurane: which LFP-tracker ↔ spiking relationships persist, emerge or disappear?

Same catalog as `relationships_awake.ipynb`, rerun on stimulation-free isoflurane maintenance (22–34 min per
recording) **with every reference taken from the awake state**: z-scoring, units, the tracker motif, and the + / −
ensembles (`src/iso_state.py`). Running, pupil **and dose** are covariates (all three are confounded:
experimenters adjusted isoflurane from live pupil readouts). Same mouse split as awake; the whole catalog was
tested in discovery, BH-significant relationships were tested in confirmation (one-sided, BH).

Coverage: recordings with both isoflurane and an awake tracker motif. Discovery 703063, 720762 (10 signals);
confirmation 703064, 703065, 709401 (25 signals). 688277 and 692644 (6-min awake, no awake tracker motif) and the
psilocybin session are not included.

**Status of each relationship**: *persists* = confirmed awake and under isoflurane; *emerges* = confirmed only
under isoflurane; *lost* = confirmed awake, not under isoflurane; *none* = confirmed in neither.

**Membership across states.** For each neuron: + / − / none awake vs under isoflurane, on stretches of equal length
(half the awake data) so the longer isoflurane recording cannot inflate coupling. Turnover across states is
compared with turnover within each state (two awake halves; two isoflurane stretches).

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import awake_grounding as ag
PA, PI = REPO / "data/processed/relationships_awake", REPO / "data/processed/relationships_iso"
OUT = REPO / "data" / "out" / "2026-11-15_relationships_iso"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "0.4", "axes.labelcolor": "0.15", "xtick.color": "0.3", "ytick.color": "0.3"})
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
AD, AC, AM = [pd.read_pickle(PA / f) for f in ("discovery_pooled.pkl", "confirmation_pooled.pkl", "confirmation_tests.pkl")]
ID, IC, IM = [pd.read_pickle(PI / f) for f in ("discovery_pooled.pkl", "confirmation_pooled.pkl", "confirmation_tests.pkl")]
conf = lambda M: dict(zip(M.relationship, M["confirmed (BH)"]))
ca, ci = conf(AM), conf(IM)
rels = AD.relationship.tolist()
T = []
for rel in rels:
    a, i = bool(ca.get(rel, False)), bool(ci.get(rel, False))
    g = lambda df: df.set_index("relationship")["excess over null"].get(rel, np.nan)
    m = lambda df: df.set_index("relationship")["mice same sign"].get(rel, "")
    T.append({"relationship": rel, "awake discovery": g(AD), "awake confirmation": g(AC), "awake mice": m(AC),
              "iso discovery": g(ID), "iso confirmation": g(IC), "iso mice": m(IC),
              "confirmed awake": a, "confirmed iso": i,
              "status": ("persists" if np.sign(g(AC)) == np.sign(g(IC)) else "reverses sign") if a and i
              else "emerges under iso" if i else "lost under iso" if a else "none"})
T = pd.DataFrame(T)
order = {"persists": 0, "reverses sign": 1, "emerges under iso": 2, "lost under iso": 3, "none": 4}
T = T.sort_values(["status", "relationship"], key=lambda c: c.map(order) if c.name == "status" else c).reset_index(drop=True)
T.to_csv(OUT / "1_status_per_relationship.csv", index=False)
T.round(3)

,relationship,awake discovery,awake confirmation,awake mice,iso discovery,iso confirmation,iso mice,confirmed awake,confirmed iso,status
0,R1 baseline~minus rate,0.085,0.069,2/3,0.315,0.120,3/3,True,True,persists
1,R12 R2 gain over pupil+running (contrast),0.146,0.157,3/3,0.049,0.025,3/3,True,True,persists
2,R12 R2 gain over pupil+running (pop),0.160,0.091,3/3,0.339,0.273,3/3,True,True,persists
3,R5 separation ~ coupling strength,0.618,0.508,3/3,0.444,0.528,3/3,True,True,persists
4,R1 baseline~pop rate,-0.089,-0.027,2/3,0.408,0.113,2/3,True,True,reverses sign
5,R1 baseline~plus rate,-0.032,-0.128,2/3,0.389,0.111,2/3,False,True,emerges under iso
6,R1w baseline ~ population rate,-0.062,-0.071,2/3,0.615,0.118,2/3,False,True,emerges under iso
7,R7 spread ~ dimensionality,0.096,-0.253,3/3,-0.438,-0.223,3/3,False,True,emerges under iso
8,R10 umbrella (Mantel r),0.483,0.320,3/3,0.183,0.237,3/3,True,False,lost under iso
9,R2 neurons whose motif coupling changes with b...,0.182,0.143,3/3,0.218,0.072,3/3,True,False,lost under iso


## 1. Awake vs isoflurane, every relationship
Excess over the null in the confirmation mice (filled = confirmed) and the discovery mice (open), awake (blue) and
isoflurane (red).

In [2]:
STC = {"persists": "#1baf7a", "reverses sign": "#eda100", "emerges under iso": "#e34948", "lost under iso": "#2a78d6", "none": "0.6"}
fig, ax = plt.subplots(figsize=(11, 0.45 * len(T) + 1.4))
for k, row in T.iterrows():
    for dy, st, col in ((-0.14, "awake", "#2a78d6"), (0.14, "iso", "#e34948")):
        ax.scatter(row[f"{st} discovery"], k + dy, s=30, facecolor="white", edgecolor=col, lw=1.1, zorder=3)
        ax.scatter(row[f"{st} confirmation"], k + dy, s=34, color=col if row[f"confirmed {st}"] else "white",
                   edgecolor=col, lw=1.1, marker="D", zorder=3)
        ax.plot([0, row[f"{st} confirmation"]], [k + dy] * 2, color=col, lw=2.5, alpha=0.35)
    ax.text(1.01, k, row.status, transform=ax.get_yaxis_transform(), fontsize=8, va="center", color=STC[row.status])
ax.axvline(0, color="0.5", lw=0.8); ax.set_yticks(range(len(T)), T.relationship, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("excess over null (r, R², share or slope)")
ax.set_title("Blue = awake, red = isoflurane; circle = discovery mice, diamond = confirmation mice (filled = confirmed)", fontsize=9, loc="left")
save(fig, "1_awake_vs_iso"); plt.show()

## 2. Do neurons join and leave the motif's ensembles between states?
Per signal: turnover of + / − / none labels within awake, within isoflurane, and across states (equal-length
stretches). Raw rates and with the population mean removed.

In [3]:
rows = []
for g in ("discovery", "confirmation"):
    for k, o in ag.load(PI / f"{g}.pkl").items():
        for kind in ("raw", "population removed"):
            rows.append({"group": g, "signal": k, "mouse": k[:6], "rates": kind, **{c: o["cross"][kind][c] for c in
                         ("within_awake", "within_iso", "across", "sign_kept_share", "coupling_r_across_neurons",
                          "share_coupled_awake_half", "share_coupled_iso_block")}})
X = pd.DataFrame(rows); X.to_csv(OUT / "2_membership_turnover_per_signal.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, kind in zip(axes, ("raw", "population removed")):
    d = X[X.rates == kind]
    for row in d.itertuples():
        ax.plot([0, 1, 2], [row.within_awake, row.within_iso, row.across], color="0.85", lw=0.7)
    for j, (c, col) in enumerate((("within_awake", "#2a78d6"), ("within_iso", "#e34948"), ("across", "#6250d6"))):
        ax.scatter(np.full(len(d), j), d[c], s=14, color=col, zorder=2)
        ax.plot([j - 0.3, j + 0.3], [d[c].median()] * 2, color="0.1", lw=2, zorder=3)
    wmax = d[["within_awake", "within_iso"]].max(1)
    ax.set_title(f"{kind}: across > both within in {(d.across > wmax).sum()}/{len(d)} signals "
                 f"(Wilcoxon p = {wilcoxon(d.across, wmax).pvalue:.1e})", fontsize=9, loc="left")
    ax.set_xticks([0, 1, 2], ["within awake", "within isoflurane", "awake vs isoflurane"])
axes[0].set_ylabel("share of neurons whose label changes")
save(fig, "2_membership_turnover"); plt.show()
S2 = X.groupby(["group", "rates"]).agg(signals=("across", "size"), within_awake=("within_awake", "median"),
                                       within_iso=("within_iso", "median"), across=("across", "median"),
                                       sign_kept=("sign_kept_share", "median"),
                                       coupling_r_across_neurons=("coupling_r_across_neurons", "median"))
S2.to_csv(OUT / "2_membership_turnover_summary.csv"); S2.round(3)

signals  within_awake  within_iso  across  \
group        rates                                                           
confirmation population removed       25         0.182       0.242   0.455   
             raw                      25         0.185       0.444   0.719   
discovery    population removed       10         0.116       0.284   0.415   
             raw                      10         0.081       0.322   0.565   

                                 sign_kept  coupling_r_across_neurons  
group        rates                                                     
confirmation population removed      0.517                      0.046  
             raw                     0.455                      0.021  
discovery    population removed      0.806                      0.285  
             raw                     0.796                      0.300

## 3. Is turnover just neurons slowing down?
Firing falls about 4-fold under isoflurane, and a slower neuron is harder to detect as coupled. Categories per
neuron (awake → isoflurane): stayed (same pole), flipped (other pole), left, joined, never coupled; with the
isoflurane / awake rate ratio, and the same categories among neurons whose rate did **not** drop.

In [4]:
P = pd.read_pickle(PI / "cross_state_per_neuron.pkl")
rows = []
for kind in ("raw", "population removed"):
    a, i = P[f"label_awake ({kind})"], P[f"label_iso ({kind})"]
    cat = pd.Series(np.select([(a != 0) & (i == 0), (a == 0) & (i != 0), (a != 0) & (i != 0) & (np.sign(a) != np.sign(i)),
                               (a != 0) & (i != 0)], ["left", "joined", "flipped", "stayed"], "never"), index=P.index)
    for subset, m in (("all neurons", np.ones(len(P), bool)), ("rate did not drop", (P.log_rate_ratio >= 0).to_numpy())):
        vc = cat[m].value_counts(normalize=True)
        rows.append({"rates": kind, "neurons": subset, "n": int(m.sum()),
                     **{c: vc.get(c, 0.0) for c in ("stayed", "flipped", "left", "joined", "never")},
                     **{f"rate ratio ({c})": float(2 ** P.log_rate_ratio[m & (cat == c).to_numpy()].median()) for c in ("stayed", "left", "joined")}})
S3 = pd.DataFrame(rows); S3.to_csv(OUT / "3_who_joins_and_leaves.csv", index=False)
S3.round(3)

,rates,neurons,n,stayed,flipped,left,joined,never,rate ratio (stayed),rate ratio (left),rate ratio (joined)
0,raw,all neurons,1337,0.218,0.182,0.156,0.355,0.089,0.361,0.209,0.260
1,raw,rate did not drop,119,0.328,0.076,0.227,0.261,0.109,1.600,1.495,1.488
2,population removed,all neurons,1337,0.145,0.087,0.274,0.191,0.303,0.258,0.219,0.273
3,population removed,rate did not drop,119,0.218,0.050,0.261,0.202,0.269,1.714,1.407,1.687


## Conclusions
**Persist across states (confirmed awake and under isoflurane, same direction):**
- **R5 pole separation ↔ coupling strength** (awake +0.51, isoflurane +0.53; all mice). The strongest and most
  stable theme — but it still needs the non-motif control (more LFP variance alone raises correlations).
- **R12 tracker outputs explain firing beyond pupil, running (and dose)**: population rate +0.09 R² awake →
  +0.27 under isoflurane; ensemble contrast +0.16 → +0.02 (persists but nearly vanishes).
- **R1 baseline ↔ − ensemble rate** (small, positive in both).

**Reverses:** R1 baseline ↔ population rate (slightly negative awake, positive under isoflurane).

**Emerges under isoflurane:** the baseline tracks firing (+ ensemble, population; also at the 2-min window level),
and **R7 spread ↔ dimensionality** (larger LFP swings ↔ lower-dimensional, more collective firing; −0.22,
3/3 confirmation mice). Not pre-registered for confirmation but strong in the confirmation mice (exploratory):
spread ↔ synchrony (+0.25, 3/3).

**Lost or weakened under isoflurane:** R4 intensity (the + minus − contrast no longer follows expression in the
confirmation mice); R2 (baseline-dependent coupling: same direction in 3/3 mice but misses correction, p = 0.046);
R10 umbrella was not carried to confirmation (discovery p = 0.036 missed correction) yet is strong there
(+0.24, 3/3) — treat as likely persisting, unconfirmed.

**Membership across states (the join/leave hypothesis):** between awake and isoflurane, 70 % of neurons change
label (raw) vs 16 % within awake and 41 % within isoflurane; 45 % vs 17 % / 25 % with the population mean removed
(across > both within in 27/35 signals, p = 3e-4). Firing falls ~4-fold and leavers slowed more, but among neurons
whose rate did not drop, 26 % still left and 20 % joined: turnover is not just detectability. In the confirmation
mice the awake mapping is essentially reset (sign kept in ~50 % of neurons coupled in both; across-neuron coupling
r ≈ 0); in the discovery mice it partly survives (80 %; r ≈ 0.3).